In [15]:
#Assigning short term memory to the agent
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    name:str
    messages : Annotated[list[BaseMessage], add_messages]

from langchain_core.messages import AIMessage

def chatbot(state:AgentState):
    name = state["name"]

    return {
        "messages" :[
            AIMessage(content=f"Hello {name}, I remember you.")
        ]
    }

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
builder = StateGraph(AgentState)

builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)


checkpointer = MemorySaver()

graph = builder.compile(
    checkpointer = checkpointer
)

config = {
    "configurable" : {
        "thread_id" : "thread_101"
    }
}
result = graph.invoke(
   {
        "name" : "Aniket",
           "messages" : [
               ("user", "My name is Aniket")
           ]
   },
   config
)
print(result)

{'name': 'Aniket', 'messages': [HumanMessage(content='My name is Aniket', additional_kwargs={}, response_metadata={}, id='0fc03bbc-eaf7-49d3-9eef-3a3bf983a2de'), AIMessage(content='Hello Aniket, I remember you.', additional_kwargs={}, response_metadata={}, id='52ae6b4d-ad1a-479e-98c1-2eb557f6696e', tool_calls=[], invalid_tool_calls=[])]}


In [16]:
result = graph.invoke(
    {
        "name": "Aniket",
        "messages": [
            ("user", "What is my name?")
        ]
    },
    config
    
)

print(result)

{'name': 'Aniket', 'messages': [HumanMessage(content='My name is Aniket', additional_kwargs={}, response_metadata={}, id='0fc03bbc-eaf7-49d3-9eef-3a3bf983a2de'), AIMessage(content='Hello Aniket, I remember you.', additional_kwargs={}, response_metadata={}, id='52ae6b4d-ad1a-479e-98c1-2eb557f6696e', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='d9a48251-9174-4251-b827-56587e38dc59'), AIMessage(content='Hello Aniket, I remember you.', additional_kwargs={}, response_metadata={}, id='c05605ba-709a-4021-9fb0-4549f93d81eb', tool_calls=[], invalid_tool_calls=[])]}
